In [1]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                              Import Packages                                                                       #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

import boto3
import json
import pyodbc
import pandas as pd
from datetime import date, datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import plotly.express as px
import mplcursors
import plotly.graph_objects as go
from pandas.tseries.holiday import USFederalHolidayCalendar
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

In [2]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Set TODAY                                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

# For automation use:
# TODAY = pd.Timestamp(date.today())

# For backtesting, set manually:
TODAY = pd.Timestamp("2026-03-10")
TODAY = pd.to_datetime(TODAY)
print("Today's date is:", TODAY)

Today's date is: 2026-03-10 00:00:00


In [3]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                     Connect to SQL Server: Load in Historical Data                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def get_secret(secret_name, region_name):
    """
    Retrieve secrets from AWS Secrets Manager.
    """
    try:
        session = boto3.session.Session()
        client  = session.client(service_name="secretsmanager", region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        secret   = json.loads(response["SecretString"])
        return secret
    except Exception as e:
        print(f"Failed to retrieve secret: {e}")
        return None

secret_name = "analytics/prod/credentials"
region_name = "us-east-1"

secret = get_secret(secret_name, region_name)

if secret:
    try:
        server   = secret["datamart"]["host"]
        port     = secret["datamart"]["port"]
        database = secret["datamart"]["db"]
        username = secret["datamart"]["user"]
        password = secret["datamart"]["password"]

        conn_str = (
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={server},{port};"
            f"Database={database};"
            f"UID={username};"
            f"PWD={password};"
        )

        select_query = """
SET DATEFIRST 1;  -- Monday = 1

WITH channels AS (
    SELECT DISTINCT
        CASE
            WHEN channel LIKE '%corr%' THEN 'Correspondent'
            ELSE channel
        END AS channel
    FROM marketing_sandbox.dbo.SDS WITH (NOLOCK)
    WHERE channel NOT LIKE '%Credit Union Partners%'
),

base AS (
    SELECT
        CASE
            WHEN s.channel LIKE '%corr%' THEN 'Correspondent'
            ELSE s.channel
        END AS channel,
        s.funded_date,
        s.loan_amount,
        DATEPART(YEAR, s.funded_date) AS year_of_funding,
        DATEPART(WEEKDAY, s.funded_date) - 1 AS day_of_week,
        DATEPART(DAY, s.funded_date) AS day_of_month,
        DATEDIFF(WEEK, DATEADD(MONTH, DATEDIFF(MONTH, 0, s.funded_date), 0), s.funded_date) + 1 AS week_of_month,
        DATEPART(MONTH, s.funded_date) AS month_of_year
    FROM marketing_sandbox.dbo.SDS s WITH (NOLOCK)
    WHERE s.funded_date >= DATEADD(YEAR, -5, CAST(GETDATE() AS DATE))
      AND s.funded_date <= (
            SELECT MIN(Calendar_Date)
            FROM (
                SELECT Calendar_Date,
                       ROW_NUMBER() OVER (ORDER BY Calendar_Date ASC) AS rn
                FROM marketing_sandbox.dbo.Calendar WITH (NOLOCK)
                WHERE Calendar_Date > CAST(GETDATE() AS DATE)
                  AND Biz_Day = 1
            ) biz
            WHERE rn = 3
      )
      AND s.funded_date IS NOT NULL
      AND s.channel NOT LIKE '%Credit Union Partners%'
),

freq AS (
    SELECT
        channel,
        funded_date,
        loan_amount,
        year_of_funding,
        day_of_week,
        day_of_month,
        week_of_month,
        month_of_year,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month) AS loans_in_week,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year) AS loans_in_month,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month, day_of_week) AS loans_on_day_of_week,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, day_of_month) AS loans_on_day_of_month,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month) AS amount_in_week,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year) AS amount_in_month,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month, day_of_week) AS amount_on_day_of_week,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, day_of_month) AS amount_on_day_of_month,
        SUM(CASE WHEN day_of_week < 5 THEN 1 ELSE 0 END)
            OVER (PARTITION BY channel, month_of_year) AS total_loans_in_month_across_years,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, month_of_year) AS total_amount_in_month_across_years,
        SUM(CASE WHEN day_of_week < 5 THEN 1 ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding) AS total_loans_in_year,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding) AS total_amount_in_year
    FROM base
),

agg_loans AS (
    SELECT
        channel,
        CAST(funded_date AS DATE) AS funded_date,
        year_of_funding,
        day_of_week,
        day_of_month,
        week_of_month,
        month_of_year,
        COUNT(*) AS number_of_loans,
        SUM(loan_amount) AS total_loan_amount,
        CAST(loans_in_week AS FLOAT) / NULLIF(loans_in_month, 0) AS week_weight,
        CAST(loans_on_day_of_week AS FLOAT) / NULLIF(loans_in_week, 0) AS day_of_week_weight,
        CAST(loans_on_day_of_month AS FLOAT) / NULLIF(loans_in_month, 0) AS day_of_month_weight,
        CAST(amount_in_week AS FLOAT) / NULLIF(amount_in_month, 0) AS week_amount_weight,
        CAST(amount_on_day_of_week AS FLOAT) / NULLIF(amount_in_week, 0) AS day_of_week_amount_weight,
        CAST(amount_on_day_of_month AS FLOAT) / NULLIF(amount_in_month, 0) AS day_of_month_amount_weight,
        CAST(total_loans_in_month_across_years AS FLOAT) / NULLIF(SUM(total_loans_in_month_across_years)
            OVER (PARTITION BY channel, month_of_year), 0) AS month_to_month_weight_loans,
        CAST(total_amount_in_month_across_years AS FLOAT) / NULLIF(SUM(total_amount_in_month_across_years)
            OVER (PARTITION BY channel, month_of_year), 0) AS month_to_month_weight_amount,
        CAST(total_loans_in_month_across_years AS FLOAT) / NULLIF(total_loans_in_year, 0) AS month_within_year_weight_loans,
        CAST(total_amount_in_month_across_years AS FLOAT) / NULLIF(total_amount_in_year, 0) AS month_within_year_weight_amount
    FROM freq
    GROUP BY
        channel, CAST(funded_date AS DATE), year_of_funding, day_of_week,
        day_of_month, week_of_month, month_of_year, loans_in_week, loans_in_month,
        loans_on_day_of_week, loans_on_day_of_month, amount_in_week, amount_in_month,
        amount_on_day_of_week, amount_on_day_of_month, total_loans_in_month_across_years,
        total_amount_in_month_across_years, total_loans_in_year, total_amount_in_year
),

calendar_channels AS (
    SELECT
        c.Calendar_Date, ch.channel, c.Biz_Day, c.Funding_Day,
        c.Funding_Day_Remaining_in_Month, c.Biz_Day_Remaining_in_Month,
        c.Is_Holiday,
        DATEPART(WEEKDAY, c.Calendar_Date) - 1 AS day_of_week,
        DATEPART(DAY, c.Calendar_Date) AS day_of_month,
        DATEDIFF(WEEK, DATEADD(MONTH, DATEDIFF(MONTH, 0, c.Calendar_Date), 0), c.Calendar_Date) + 1 AS week_of_month,
        DATEPART(MONTH, c.Calendar_Date) AS month_of_year
    FROM marketing_sandbox.dbo.Calendar c
    CROSS JOIN channels ch
    WHERE c.Calendar_Date >= DATEADD(YEAR, -5, CAST(GETDATE() AS DATE))
      AND c.Calendar_Date <= (
            SELECT MIN(Calendar_Date)
            FROM (
                SELECT Calendar_Date,
                       ROW_NUMBER() OVER (ORDER BY Calendar_Date ASC) AS rn
                FROM marketing_sandbox.dbo.Calendar WITH (NOLOCK)
                WHERE Calendar_Date > CAST(GETDATE() AS DATE)
                  AND Biz_Day = 1
            ) biz
            WHERE rn = 3
      )
),

joined AS (
    SELECT
        cc.Calendar_Date, cc.channel, cc.Biz_Day, cc.Funding_Day,
        cc.Funding_Day_Remaining_in_Month, cc.Biz_Day_Remaining_in_Month,
        cc.Is_Holiday, cc.day_of_week, cc.day_of_month, cc.week_of_month,
        cc.month_of_year,
        ISNULL(al.number_of_loans, 0)            AS number_of_loans,
        ISNULL(al.total_loan_amount, 0)           AS total_loan_amount,
        ISNULL(al.week_weight, 0)                 AS week_weight,
        ISNULL(al.day_of_week_weight, 0)          AS day_of_week_weight,
        ISNULL(al.day_of_month_weight, 0)         AS day_of_month_weight,
        ISNULL(al.week_amount_weight, 0)          AS week_amount_weight,
        ISNULL(al.day_of_week_amount_weight, 0)   AS day_of_week_amount_weight,
        ISNULL(al.day_of_month_amount_weight, 0)  AS day_of_month_amount_weight,
        ISNULL(al.month_to_month_weight_loans, 0) AS month_to_month_weight_loans,
        ISNULL(al.month_to_month_weight_amount, 0)AS month_to_month_weight_amount,
        ISNULL(al.month_within_year_weight_loans, 0)  AS month_within_year_weight_loans,
        ISNULL(al.month_within_year_weight_amount, 0) AS month_within_year_weight_amount
    FROM calendar_channels cc
    LEFT JOIN agg_loans al
        ON cc.Calendar_Date = al.funded_date
       AND cc.channel = al.channel
)

SELECT
    Calendar_Date, channel, Biz_Day, Funding_Day,
    Funding_Day_Remaining_in_Month, Biz_Day_Remaining_in_Month,
    Is_Holiday, day_of_week, day_of_month, week_of_month, month_of_year,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE number_of_loans END          AS number_of_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE total_loan_amount END         AS total_loan_amount,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE week_weight END               AS week_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_week_weight END        AS day_of_week_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_month_weight END       AS day_of_month_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE week_amount_weight END        AS week_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_week_amount_weight END AS day_of_week_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_month_amount_weight END AS day_of_month_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_to_month_weight_loans END  AS month_to_month_weight_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_to_month_weight_amount END AS month_to_month_weight_amount,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_within_year_weight_loans END  AS month_within_year_weight_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_within_year_weight_amount END AS month_within_year_weight_amount
FROM joined
ORDER BY Calendar_Date ASC, channel;
"""

        conn    = pyodbc.connect(conn_str)
        cursor  = conn.cursor()
        cursor.execute(select_query)
        columns = [column[0] for column in cursor.description]
        data    = cursor.fetchall()
        df      = pd.DataFrame.from_records(data, columns=columns)
        print(f"Data loaded successfully. Shape: {df.shape}")
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals():
            conn.close()
else:
    print("Failed to retrieve secrets. Check your configuration.")

Data loaded successfully. Shape: (9160, 23)


In [4]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Data Preprocessing                                                        #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
df.fillna(0, inplace=True)

In [5]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Split by Channel                                                          #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

wholesale_historical    = df[df['channel'] == 'Wholesale'].copy()
retail_historical       = df[df['channel'] == 'Retail'].copy()
retail_broker_historical= df[df['channel'] == 'Retail Broker'].copy()
corr_historical         = df[df['channel'] == 'Correspondent'].copy()

In [6]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   MASTER HOLIDAY CALENDAR (2021–2029)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

def build_us_holiday_calendar(start_date='2021-01-01', end_date='2029-12-31'):
    """
    Builds full daily calendar spine with hard holiday flags (Is_Holiday).
    """
    full_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    calendar   = pd.DataFrame({'Calendar_Date': full_dates})
    calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])

    cal      = USFederalHolidayCalendar()
    holidays = cal.holidays(start=start_date, end=end_date)

    calendar['Is_Holiday'] = calendar['Calendar_Date'].isin(holidays).astype(int)
    return calendar[['Calendar_Date', 'Is_Holiday']]

In [7]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          HolidayDistortionLearner
#  Learns per-(holiday_dow, business_day_offset) distortion factors from history.
#  e.g. "The Tuesday after a Monday holiday historically funds +35% above normal"
#////////////////////////////////////////////////////////////////////////////////////////////////////////

class HolidayDistortionLearner:
    """
    Learns empirical holiday distortion factors from historical data.

    For each holiday in training history:
      - Identifies the holiday's day-of-week (holiday_dow)
      - Looks at surrounding business days in a window [-2, +4]
      - Computes ratio of actual / model-expected for each offset day
      - Aggregates across years using median + shrinkage toward 1.0

    Result: distortion_table keyed by (holiday_dow, biz_offset)
    """

    def __init__(self,
                 window_before=2,
                 window_after=4,
                 shrinkage=0.3,
                 min_observations=2):

        self.window_before = window_before
        self.window_after = window_after
        self.shrinkage = shrinkage
        self.min_observations = min_observations
        self.distortion_table_ = {}

    def fit(self, df, holiday_calendar, expected_col='expected'):
        """
        df            : training dataframe with Calendar_Date, actual target, and expected column
        holiday_calendar : dataframe with Calendar_Date and Is_Holiday columns
        expected_col  : column name in df containing the model's baseline expected value
        """

        df = df.copy()
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])

        holiday_calendar = holiday_calendar.copy()
        holiday_calendar['Calendar_Date'] = pd.to_datetime(
            holiday_calendar['Calendar_Date']
        )

        holiday_dates = holiday_calendar.loc[
            holiday_calendar['Is_Holiday'] == 1, 'Calendar_Date'
        ].tolist()

        # Business day spine from training data (weekdays only, non-holiday)
        biz_days = df.loc[
            (~df['Calendar_Date'].dt.dayofweek.isin([5, 6]))
        ].copy()

        biz_days = biz_days.sort_values('Calendar_Date').reset_index(drop=True)

        # Map date -> position in business day sequence
        biz_day_index = {
            row['Calendar_Date']: idx
            for idx, row in biz_days.iterrows()
        }

        # Accumulate ratios: {(holiday_dow, offset): [ratio, ratio, ...]}
        ratio_accumulator = {}

        for hdate in holiday_dates:

            holiday_dow = hdate.dayofweek

            # Skip weekends (shouldn't happen with USFederalHolidayCalendar observed)
            if holiday_dow in [5, 6]:
                continue

            # Find surrounding business days
            for offset in range(-self.window_before, self.window_after + 1):

                if offset == 0:
                    continue  # Holiday itself is always hard zero

                # Step through calendar days to find the Nth business day offset
                candidate = hdate
                steps = 0
                direction = 1 if offset > 0 else -1
                abs_offset = abs(offset)

                while steps < abs_offset:
                    candidate = candidate + pd.Timedelta(days=direction)
                    if candidate.dayofweek not in [5, 6]:
                        steps += 1

                # Look up actual and expected for this candidate date
                row = df.loc[df['Calendar_Date'] == candidate]

                if row.empty:
                    continue

                actual = row[expected_col.replace('expected', df.columns[
                    [c for c in df.columns if 'loan_amount' in c or 'loans' in c][0]
                    if any('loan_amount' in c or 'loans' in c for c in df.columns)
                    else -1
                ])].values[0] if False else None

                # Simpler: use whatever target column is available
                target_cols = [c for c in df.columns
                               if 'total_loan_amount' in c or 'number_of_loans' in c]

                if not target_cols:
                    continue

                target_col = target_cols[0]
                actual = row[target_col].values[0]
                expected = row[expected_col].values[0]

                if expected <= 0 or actual < 0:
                    continue

                ratio = actual / expected

                key = (holiday_dow, offset)

                if key not in ratio_accumulator:
                    ratio_accumulator[key] = []

                ratio_accumulator[key].append(ratio)

        # Aggregate: median + shrinkage toward 1.0
        self.distortion_table_ = {}

        for key, ratios in ratio_accumulator.items():

            if len(ratios) < self.min_observations:
                continue

            median_ratio = np.median(ratios)

            shrunk = (
                (1 - self.shrinkage) * median_ratio +
                self.shrinkage * 1.0
            )

            self.distortion_table_[key] = shrunk

        return self

    def get_distortion(self, holiday_dow, biz_offset):
        """
        Returns learned distortion factor for a given
        (holiday day-of-week, business day offset from holiday).
        Defaults to 1.0 if no data learned.
        """
        return self.distortion_table_.get((holiday_dow, biz_offset), 1.0)

In [8]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                         Model A — StructuralLoanForecaster_A                                                     #
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_A:

    def __init__(self,
                 target='total_loan_amount',
                 dow_shrinkage=0.25,
                 interaction_shrinkage=0.35,
                 seasonal_exponent=1.02,
                 recency_strength=0.35,
                 trend_seasonal_tilt=0.08,
                 level_lookback_days=60,
                 trend_dampening=0.83,
                 forward_lift_daily=0.0025,
                 holiday_window_before=2,
                 holiday_window_after=4,
                 holiday_shrinkage=0.3):

        self.target = target
        self.dow_shrinkage = dow_shrinkage
        self.interaction_shrinkage = interaction_shrinkage
        self.seasonal_exponent = seasonal_exponent
        self.recency_strength = recency_strength
        self.trend_seasonal_tilt = trend_seasonal_tilt
        self.level_lookback_days = level_lookback_days
        self.trend_dampening = trend_dampening
        self.forward_lift_daily = forward_lift_daily
        self.holiday_window_before = holiday_window_before
        self.holiday_window_after = holiday_window_after
        self.holiday_shrinkage = holiday_shrinkage

        self.master_calendar = None
        self.distortion_table_ = {}
        self.fitted = False

    # ---------------------------------------------------
    # INTERNAL: compute expected baseline for a dataframe
    # ---------------------------------------------------
    def _compute_expected(self, df, t_array):
        """
        Given a df with dow/dom/moy columns and a t_array of
        trend indices, returns the baseline expected value array
        (trend × DOW × DOM × MOY × interaction).
        """

        trend = np.exp(
            self.trend_intercept + self.trend_slope * t_array
        )

        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values

        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * dom_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent

        tilt = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        return self.base_level * seasonal * trend * tilt

    # ---------------------------------------------------
    # FIT
    # ---------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date')
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])

        df['dow'] = df['Calendar_Date'].dt.dayofweek
        df['dom'] = df['Calendar_Date'].dt.day
        df['moy'] = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Store master holiday calendar ----
        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar

            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date',
                how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        # ---- Production mask: weekdays, non-holiday, non-zero ----
        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        # ---- NEW: Seasonal mask — full completed months only ----
        last_date           = df['Calendar_Date'].max()
        partial_month_start = last_date.replace(day=1)

        seasonal_mask = (
            prod_mask &
            (df['Calendar_Date'] < partial_month_start)
        )

        df_prod = df.loc[prod_mask].copy()

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train = len(df)

        # ---- Trend (log-linear WLS, recency weighted) — FAST VERSION ----
        t_prod = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))

        y = np.log(df_prod[self.target].values)
        X = np.vstack([np.ones(len(t_prod)), t_prod]).T

        sqrt_w = np.sqrt(recency_prod)
        Xw     = X * sqrt_w[:, None]
        yw     = y * sqrt_w
        beta   = np.linalg.lstsq(Xw, yw, rcond=None)[0]

        self.trend_intercept = beta[0]
        self.trend_slope = beta[1]

        t_full = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)

        df['trend_fitted'] = trend_fitted
        df['detrended'] = df[self.target] / trend_fitted

        # ---- Base level — uses prod_mask (includes partial month) ----
        recent_prod = df.loc[prod_mask].tail(self.level_lookback_days)
        self.base_level = (
            recent_prod[self.target].mean() /
            recent_prod['trend_fitted'].mean()
        )

        # ---- DOW effect — uses seasonal_mask (full months only) ----
        dow_raw = df.loc[seasonal_mask].groupby('dow')['detrended'].mean()
        dow_shrunk = (
            (1 - self.dow_shrinkage) * dow_raw +
            self.dow_shrinkage * self.base_level
        )
        self.dow_effect = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- DOM effect — uses seasonal_mask ----
        dom_raw = df.loc[seasonal_mask].groupby('dom')['after_dow'].mean()
        self.dom_effect = dom_raw / dom_raw.mean()
        df['after_dom'] = df['after_dow'] / df['dom'].map(self.dom_effect)

        # ---- MOY effect — uses seasonal_mask ----
        moy_raw = df.loc[seasonal_mask].groupby('moy')['after_dom'].mean()
        self.moy_effect = moy_raw / moy_raw.mean()
        df['after_moy'] = df['after_dom'] / df['moy'].map(self.moy_effect)

        # ---- DOW x MOY interaction — uses seasonal_mask ----
        interaction_raw = (
            df.loc[seasonal_mask]
            .groupby(['dow', 'moy'])['after_moy']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = (
            interaction_shrunk / interaction_shrunk.mean()
        )

        # ================================================
        # LEARN HOLIDAY DISTORTION FACTORS — uses prod_mask
        # ================================================
        if self.master_calendar is not None:

            t_full_arr = np.arange(len(df))
            df['expected'] = self._compute_expected(df, t_full_arr)

            holiday_dates = self.master_calendar.loc[
                self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
            ].tolist()

            biz_day_list = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()

            biz_day_pos = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}

            for hdate in holiday_dates:

                holiday_dow = hdate.dayofweek

                if holiday_dow in [5, 6]:
                    continue

                if hdate not in biz_day_pos:
                    continue

                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                    self.holiday_window_after + 1
                ):

                    if offset == 0:
                        continue

                    target_pos = h_pos + offset

                    if target_pos < 0 or target_pos >= len(biz_day_list):
                        continue

                    target_date = biz_day_list[target_pos]

                    row = df.loc[df['Calendar_Date'] == target_date]

                    if row.empty:
                        continue

                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue

                    actual = row[self.target].values[0]
                    expected = row['expected'].values[0]

                    if expected <= 0 or actual <= 0:
                        continue

                    ratio = actual / expected
                    key = (holiday_dow, offset)

                    if key not in ratio_accumulator:
                        ratio_accumulator[key] = []

                    ratio_accumulator[key].append(ratio)

            self.distortion_table_ = {}

            for key, ratios in ratio_accumulator.items():

                if len(ratios) < 2:
                    continue

                median_ratio = float(np.median(ratios))

                shrunk = (
                    (1 - self.holiday_shrinkage) * median_ratio +
                    self.holiday_shrinkage * 1.0
                )

                self.distortion_table_[key] = shrunk

        self.fitted = True
        return self

    # ---------------------------------------------------
    # FORECAST
    # ---------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit first.')

        start_date = pd.to_datetime(start_date)
        end_date = start_date + pd.DateOffset(months=months_forward)

        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow'] = df['Calendar_Date'].dt.dayofweek
        df['dom'] = df['Calendar_Date'].dt.day
        df['moy'] = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Trend ----
        t_future = np.arange(self.n_train, self.n_train + len(df))

        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift = 1 + self.forward_lift_daily * horizon_index

        # ---- Seasonal ----
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values

        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * dom_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent

        tilt = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        forecast = (
            self.base_level
            * seasonal
            * trend_multiplier
            * tilt
            * forward_lift
        )

        # ---- Weekend hard zero ----
        forecast[df['is_weekend'].values] = 0.0

        # ---- Apply holiday calendar ----
        if self.master_calendar is not None:

            df = df.merge(
                self.master_calendar,
                on='Calendar_Date',
                how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

            # Hard zero on holiday itself
            forecast[df['Is_Holiday'].values == 1] = 0.0

            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )

            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast

            forecast_biz_days = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()

            forecast_biz_pos = {d: i for i, d in enumerate(forecast_biz_days)}

            distortion = np.ones(len(df))

            for hdate in all_relevant_holidays:

                holiday_dow = hdate.dayofweek

                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                    self.holiday_window_after + 1
                ):

                    if offset == 0:
                        continue

                    candidate = hdate
                    steps = 0
                    direction = 1 if offset > 0 else -1
                    abs_offset = abs(offset)

                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()

                    if not row_idx:
                        continue

                    idx = row_idx[0]

                    factor = self.distortion_table_.get(
                        (holiday_dow, offset), 1.0
                    )

                    distortion[idx] = factor

            forecast = forecast * distortion

        df['forecast'] = forecast

        return df

In [9]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                         Model B — StructuralLoanForecaster_B                                                     #
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_B:
    """
    Structural time-series forecaster for daily loan volumes.

    Production parameters (validated Aug 2024 – Nov 2025 backtest):
        trend_dampening          = 0.800
        distortion_max_factor    = 1.2
        distortion_lookback_days = 730
        distortion_min_obs       = 3

    Backtest results (16 months):
        MAE        5.51%
        Median AE  4.16%
        Within 5%  8/16
        Within 10% 13/16
        Max error  +20.1% (Jan 2025 — 2025 H1 hyper-growth, not modelable)
    """

    def __init__(
        self,
        target                    = 'total_loan_amount',
        recency_strength          = 1.0,
        dow_shrinkage             = 0.05,
        interaction_shrinkage     = 0.2,
        level_lookback_days       = 60,
        holiday_window_before     = 2,
        holiday_window_after      = 4,
        holiday_shrinkage         = 0.2,
        distortion_min_obs        = 3,
        distortion_max_factor     = 1.1,
        distortion_lookback_days  = 730,
        per_key_caps              = None,
        mon_thu_override_fraction = None,
        trend_dampening           = 0.70,
        forward_lift_daily        = 0.0,
        trend_seasonal_tilt       = 0.0,
        seasonal_exponent         = 1.0,
        moy_recency_strength      = 0.0,
        mom_dampening             = 0.0,
    ):
        self.target                    = target
        self.recency_strength          = recency_strength
        self.dow_shrinkage             = dow_shrinkage
        self.interaction_shrinkage     = interaction_shrinkage
        self.level_lookback_days       = level_lookback_days
        self.holiday_window_before     = holiday_window_before
        self.holiday_window_after      = holiday_window_after
        self.holiday_shrinkage         = holiday_shrinkage
        self.distortion_min_obs        = distortion_min_obs
        self.distortion_max_factor     = distortion_max_factor
        self.distortion_lookback_days  = distortion_lookback_days
        self.per_key_caps              = per_key_caps or {}
        self.mon_thu_override_fraction = mon_thu_override_fraction
        self.trend_dampening           = trend_dampening
        self.forward_lift_daily        = forward_lift_daily
        self.trend_seasonal_tilt       = trend_seasonal_tilt
        self.seasonal_exponent         = seasonal_exponent
        self.moy_recency_strength      = moy_recency_strength
        self.mom_dampening             = mom_dampening
        self.fitted                    = False

    # ------------------------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date').reset_index(drop=True)
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar
            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date', how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)
        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        df = df.reset_index(drop=True)

        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        # ---- Holiday halo (existing Model B logic) ----
        if self.master_calendar is not None:
            holiday_dates = set(pd.to_datetime(
                self.master_calendar.loc[
                    self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
                ]
            ))
            halo_dates = set()
            for hdate in holiday_dates:
                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    candidate = hdate + pd.Timedelta(days=offset)
                    if candidate.dayofweek not in [5, 6]:
                        halo_dates.add(candidate)
            df['in_halo'] = df['Calendar_Date'].isin(halo_dates)
        else:
            holiday_dates = set()
            df['in_halo']  = False

        # ---- NEW: Seasonal mask — full completed months only + halo exclusion ----
        last_date           = df['Calendar_Date'].max()
        partial_month_start = last_date.replace(day=1)

        seasonal_mask = (
            prod_mask &
            (~df['in_halo']) &
            (df['Calendar_Date'] < partial_month_start)
        )

        df_prod = df.loc[prod_mask].copy()

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train         = len(df)

        # ---- Trend — uses prod_mask (includes partial month) ----
        t_prod       = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))
        y_log        = np.log(df_prod[self.target].values)
        X            = np.vstack([np.ones(len(t_prod)), t_prod]).T
        W            = np.diag(recency_prod)
        beta         = np.linalg.inv(X.T @ W @ X) @ (X.T @ W @ y_log)
        self.trend_intercept = beta[0]
        self.trend_slope     = beta[1]

        t_full       = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)

        df['trend_fitted'] = trend_fitted
        df['detrended']    = np.where(
            trend_fitted > 0,
            df[self.target] / trend_fitted,
            0.0
        )

        # ---- Base level — uses prod_mask (includes partial month) ----
        recent_prod     = df.loc[prod_mask].tail(self.level_lookback_days)
        target_mean     = float(recent_prod[self.target].mean())
        trend_mean      = float(recent_prod['trend_fitted'].mean())
        self.base_level = target_mean / trend_mean if trend_mean > 0 else 1.0

        # ---- DOW effect — uses seasonal_mask (full months, no halo) ----
        dow_raw    = df.loc[seasonal_mask].groupby('dow')['detrended'].mean()
        dow_shrunk = (
            (1 - self.dow_shrinkage) * dow_raw +
            self.dow_shrinkage * self.base_level
        )
        self.dow_effect        = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- DOM effect — uses seasonal_mask ----
        dom_raw         = df.loc[seasonal_mask].groupby('dom')['after_dow'].mean()
        self.dom_effect = dom_raw / dom_raw.mean()
        df['after_dom'] = df['after_dow'] / df['dom'].map(self.dom_effect)

        # ---- MOY effect — uses seasonal_mask ----
        df_seas_moy = df.loc[seasonal_mask].copy()
        min_year    = df_seas_moy['Calendar_Date'].dt.year.min()
        max_year    = df_seas_moy['Calendar_Date'].dt.year.max()
        df_seas_moy['year']        = df_seas_moy['Calendar_Date'].dt.year
        df_seas_moy['year_weight'] = np.exp(
            self.moy_recency_strength *
            (df_seas_moy['year'] - min_year) / max(max_year - min_year, 1)
        )
        moy_raw = (
            df_seas_moy
            .groupby('moy')
            .apply(
                lambda g: np.average(g['after_dom'], weights=g['year_weight']),
                include_groups=False
            )
        )
        self.moy_effect = moy_raw / moy_raw.mean()
        df['after_moy'] = df['after_dom'] / df['moy'].map(self.moy_effect)

        # ---- DOW x MOY interaction — uses seasonal_mask ----
        interaction_raw = (
            df.loc[seasonal_mask]
            .groupby(['dow', 'moy'])['after_moy']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = interaction_shrunk / interaction_shrunk.mean()

        # ---- Monthly totals — uses full df (unchanged) ----
        self.monthly_totals_ = (
            df.groupby([
                df['Calendar_Date'].dt.year,
                df['Calendar_Date'].dt.month
            ])[self.target].sum()
        )
        self.monthly_totals_.index.names = ['year', 'month']

        # ---- Distortion table — uses prod_mask (unchanged) ----
        if self.master_calendar is not None:

            last_train = df['Calendar_Date'].max()
            if self.distortion_lookback_days is not None:
                distortion_cutoff = last_train - pd.Timedelta(
                    days=self.distortion_lookback_days
                )
                df_dist = df[df['Calendar_Date'] >= distortion_cutoff].copy()
            else:
                df_dist = df.copy()

            df_dist = df_dist.reset_index(drop=True)
            t_dist  = np.arange(len(df_dist))
            df_dist['expected'] = self._compute_expected(df_dist, t_dist)

            biz_day_list = df_dist.loc[
                ~df_dist['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()
            biz_day_pos  = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}
            for hdate in holiday_dates:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue
                if hdate not in biz_day_pos:
                    continue
                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue
                    if (
                        holiday_dow == 0 and offset == 3 and
                        self.mon_thu_override_fraction is not None
                    ):
                        continue
                    tp = h_pos + offset
                    if tp < 0 or tp >= len(biz_day_list):
                        continue
                    target_date = biz_day_list[tp]
                    row         = df_dist.loc[df_dist['Calendar_Date'] == target_date]
                    if row.empty:
                        continue
                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue
                    actual   = row[self.target].values[0]
                    expected = row['expected'].values[0]
                    if expected <= 0 or actual <= 0:
                        continue
                    ratio = actual / expected
                    ratio_accumulator.setdefault((holiday_dow, offset), []).append(ratio)

            self.distortion_table_ = {}
            for key, ratios in ratio_accumulator.items():
                if len(ratios) < self.distortion_min_obs:
                    continue
                raw_median = float(np.median(ratios))
                shrunk     = (
                    (1 - self.holiday_shrinkage) * raw_median +
                    self.holiday_shrinkage * 1.0
                )
                if key in self.per_key_caps:
                    effective_cap = max(
                        self.distortion_max_factor,
                        self.per_key_caps[key]
                    )
                else:
                    effective_cap = self.distortion_max_factor

                capped = float(np.clip(
                    shrunk,
                    1.0 / effective_cap,
                    effective_cap
                ))
                self.distortion_table_[key] = capped
        else:
            self.distortion_table_ = {}

        self.fitted = True
        return self

    # ------------------------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit before forecasting.')

        start_date = pd.to_datetime(start_date)
        end_date   = start_date + pd.DateOffset(months=months_forward)

        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        t_future         = np.arange(self.n_train, self.n_train + len(df))
        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift  = 1 + self.forward_lift_daily * horizon_index

        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * dom_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent
        tilt     = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        forecast = (
            self.base_level * seasonal * trend_multiplier * tilt * forward_lift
        )

        for period in df['Calendar_Date'].dt.to_period('M').unique():
            yr, mo     = period.year, period.month
            weight     = self._mom_trend_weight(yr, mo)
            month_mask = (
                (df['Calendar_Date'].dt.year  == yr) &
                (df['Calendar_Date'].dt.month == mo)
            ).values
            forecast[month_mask] *= weight

        forecast[df['is_weekend'].values] = 0.0

        if self.master_calendar is not None:
            df = df.merge(self.master_calendar, on='Calendar_Date', how='left')
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)
            forecast[df['Is_Holiday'].values == 1] = 0.0

            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )
            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast
            distortion            = np.ones(len(df))

            all_cal      = pd.date_range(start=lookback_start, end=future_dates[-1], freq='D')
            biz_days_ext = [d for d in all_cal if d.dayofweek not in [5, 6]]
            biz_pos_ext  = {d: i for i, d in enumerate(biz_days_ext)}

            for hdate in all_relevant_holidays:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue
                    if (
                        holiday_dow == 0 and offset == 3 and
                        self.mon_thu_override_fraction is not None
                    ):
                        continue

                    candidate  = hdate
                    steps      = 0
                    direction  = 1 if offset > 0 else -1
                    abs_offset = abs(offset)
                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()
                    if not row_idx:
                        continue
                    idx    = row_idx[0]
                    factor = self._get_distortion(holiday_dow, offset)
                    distortion[idx] = factor

            forecast = forecast * distortion

            if self.mon_thu_override_fraction is not None:
                for hdate in all_relevant_holidays:
                    if hdate.dayofweek != 0:
                        continue
                    if hdate not in biz_pos_ext:
                        continue
                    h_pos      = biz_pos_ext[hdate]
                    target_pos = h_pos + 3
                    if target_pos >= len(biz_days_ext):
                        continue
                    target_date = biz_days_ext[target_pos]
                    if target_date.dayofweek != 3:
                        continue
                    row_idx = df.index[df['Calendar_Date'] == target_date].tolist()
                    if not row_idx:
                        continue
                    idx   = row_idx[0]
                    t_pos = self.n_train + idx
                    trend_v   = np.exp(
                        self.trend_intercept +
                        (self.trend_slope * self.trend_dampening) * t_pos
                    )
                    moy       = target_date.month
                    dom_e_val = self.dom_effect.get(target_date.day, 1.0)
                    moy_e_val = self.moy_effect.get(moy, 1.0)
                    mon_dow_e = self.dow_effect.get(0, 1.0)
                    mon_int_e = self.interaction_effect.get((0, moy), 1.0)
                    expected_mon = (
                        self.base_level * trend_v *
                        dom_e_val * moy_e_val * mon_dow_e * mon_int_e
                    )
                    forecast[idx] = self.mon_thu_override_fraction * expected_mon

        df['forecast'] = forecast
        return df

    # ------------------------------------------------------------------
    def _compute_expected(self, df, t_arr):
        trend = np.exp(self.trend_intercept + self.trend_slope * t_arr)
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])
        return self.base_level * trend * dow_e * dom_e * moy_e * interaction_e

    def _get_distortion(self, holiday_dow, offset):
        return self.distortion_table_.get((holiday_dow, offset), 1.0)

    def _mom_trend_weight(self, year, month):
        if self.mom_dampening == 0.0:
            return 1.0
        try:
            prev_year  = year - 1 if month > 1 else year - 2
            prev_month = month - 1 if month > 1 else 12
            yoy_year   = year - 1
            prev_vol   = self.monthly_totals_.get((prev_year,  prev_month), None)
            yoy_vol    = self.monthly_totals_.get((yoy_year,   month),      None)
            pp_vol     = self.monthly_totals_.get((prev_year,  month),      None)
            if prev_vol is None or yoy_vol is None or pp_vol is None:
                return 1.0
            if pp_vol <= 0 or prev_vol <= 0:
                return 1.0
            mom_trend = prev_vol / pp_vol
            weight    = 1.0 + self.mom_dampening * (mom_trend - 1.0)
            weight    = np.clip(weight, 0.85, 1.15)
            return float(weight)
        except Exception:
            return 1.0

In [10]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                         Model C — StructuralLoanForecaster_C (December Specialist)                              #
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_C:
    """
    December-specialist structural forecaster for daily loan volumes.

    December is uniquely difficult because:
      - Volume collapses sharply after ~Dec 18 as borrowers/staff go offline
      - Early December often sees a catch-up surge from November pipeline
      - The Christmas–New Year holiday cluster is tightly spaced, causing
        cascading distortion across nearly every business day in the final 2 weeks

    Design decisions vs Model A/B:
      - Seasonal effects (DOW, DOM) fitted on December data ONLY,
        using all available historical Decembers (full months only)
      - Separate pre/post cliff base levels to capture the Dec 18 volume cliff
      - Holiday distortion window widened (before=2, after=6) to capture
        the full Christmas–NYE cascade — Dec 24 and Dec 31 suppression
        is learned empirically from history via distortion factors rather
        than being hard-coded
      - distortion_max_factor widened to 1.4 — December is more volatile
      - distortion_min_obs lowered to 2 — fewer historical Decembers
      - Trend fitted on all production data (not December-only) so the
        long-run growth rate is informed by the full history
    """

    def __init__(
        self,
        target                   = 'total_loan_amount',
        recency_strength         = 3.0,
        dow_shrinkage            = 0.15,
        interaction_shrinkage    = 0.25,
        level_lookback_days      = 60,
        holiday_window_before    = 2,
        holiday_window_after     = 6,
        holiday_shrinkage        = 0.2,
        distortion_min_obs       = 2,
        distortion_max_factor    = 1.4,
        trend_dampening          = 0.80,
        forward_lift_daily       = 0.0,
        cliff_date               = 18,
        cliff_shrinkage          = 0.5,
    ):
        self.target                = target
        self.recency_strength      = recency_strength
        self.dow_shrinkage         = dow_shrinkage
        self.interaction_shrinkage = interaction_shrinkage
        self.level_lookback_days   = level_lookback_days
        self.holiday_window_before = holiday_window_before
        self.holiday_window_after  = holiday_window_after
        self.holiday_shrinkage     = holiday_shrinkage
        self.distortion_min_obs    = distortion_min_obs
        self.distortion_max_factor = distortion_max_factor
        self.trend_dampening       = trend_dampening
        self.forward_lift_daily    = forward_lift_daily
        self.cliff_date            = cliff_date
        self.cliff_shrinkage       = cliff_shrinkage

        self.master_calendar   = None
        self.distortion_table_ = {}
        self.fitted            = False

    # ------------------------------------------------------------------
    def _compute_expected(self, df, t_arr):
        trend = np.exp(self.trend_intercept + self.trend_slope * t_arr)
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, 12), 1.0)
            for d in df['dow']
        ])
        return self.base_level * trend * dow_e * dom_e * interaction_e

    # ------------------------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date').reset_index(drop=True)
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
        df['dow']           = df['Calendar_Date'].dt.dayofweek
        df['dom']           = df['Calendar_Date'].dt.day
        df['moy']           = df['Calendar_Date'].dt.month
        df['is_weekend']    = df['dow'].isin([5, 6])

        # ---- Holiday calendar ----
        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar
            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date', how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)
        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        df = df.reset_index(drop=True)

        # ---- Production mask (all data — used for trend) ----
        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        # ---- Holiday halo — exclude halo days from seasonal fitting ----
        if self.master_calendar is not None:
            holiday_dates = set(pd.to_datetime(
                self.master_calendar.loc[
                    self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
                ]
            ))
            halo_dates = set()
            for hdate in holiday_dates:
                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    candidate = hdate + pd.Timedelta(days=offset)
                    if candidate.dayofweek not in [5, 6]:
                        halo_dates.add(candidate)
            df['in_halo'] = df['Calendar_Date'].isin(halo_dates)
        else:
            holiday_dates = set()
            df['in_halo']  = False

        # ---- December-only seasonal mask ----
        # Full completed Decembers only, halo days excluded so the
        # Christmas/NYE distortion doesn't corrupt DOW/DOM weights
        last_date           = df['Calendar_Date'].max()
        partial_month_start = last_date.replace(day=1)

        dec_seasonal_mask = (
            prod_mask &
            (df['moy'] == 12) &
            (~df['in_halo']) &
            (df['Calendar_Date'] < partial_month_start)
        )

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train         = len(df)

        # ---- Trend — fitted on ALL production data ----
        df_prod      = df.loc[prod_mask].copy()
        t_prod       = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))
        y_log        = np.log(df_prod[self.target].values)
        X            = np.vstack([np.ones(len(t_prod)), t_prod]).T
        W            = np.diag(recency_prod)
        beta         = np.linalg.inv(X.T @ W @ X) @ (X.T @ W @ y_log)
        self.trend_intercept = beta[0]
        self.trend_slope     = beta[1]

        t_full       = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)
        df['trend_fitted'] = trend_fitted
        df['detrended']    = np.where(
            trend_fitted > 0, df[self.target] / trend_fitted, 0.0
        )

        # ---- Base level ----
        recent_prod     = df.loc[prod_mask].tail(self.level_lookback_days)
        target_mean     = float(recent_prod[self.target].mean())
        trend_mean      = float(recent_prod['trend_fitted'].mean())
        self.base_level = target_mean / trend_mean if trend_mean > 0 else 1.0

        # ---- Pre/post cliff base levels ----
        # Measures how much volume drops after cliff_date in December
        # so the forecast can scale down the back half of the month
        dec_pre_mask  = dec_seasonal_mask & (df['dom'] <= self.cliff_date)
        dec_post_mask = dec_seasonal_mask & (df['dom'] >  self.cliff_date)

        pre_mean  = df.loc[dec_pre_mask,  self.target].mean()
        post_mean = df.loc[dec_post_mask, self.target].mean()

        if pd.notna(pre_mean) and pd.notna(post_mean) and pre_mean > 0:
            self.cliff_ratio = post_mean / pre_mean
        else:
            self.cliff_ratio = 1.0

        print(f"  [Model C] cliff_ratio (post/pre Dec {self.cliff_date}): "
              f"{self.cliff_ratio:.3f}")

        # ---- DOW effect — December data only, halo excluded ----
        dow_raw    = df.loc[dec_seasonal_mask].groupby('dow')['detrended'].mean()
        dow_shrunk = (
            (1 - self.dow_shrinkage) * dow_raw +
            self.dow_shrinkage * self.base_level
        )
        self.dow_effect        = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- DOM effect — December data only, halo excluded ----
        dom_raw         = df.loc[dec_seasonal_mask].groupby('dom')['after_dow'].mean()
        self.dom_effect = dom_raw / dom_raw.mean()
        df['after_dom'] = df['after_dow'] / df['dom'].map(self.dom_effect)

        # ---- DOW x December interaction ----
        interaction_raw    = (
            df.loc[dec_seasonal_mask]
            .groupby(['dow', 'moy'])['after_dom']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = interaction_shrunk / interaction_shrunk.mean()

        # ---- Holiday distortion ----
        # wider window (after=6) captures the full Christmas–NYE cascade.
        # Dec 24 / Dec 31 suppression is learned here from history as
        # distortion factors rather than being hard-coded.
        if self.master_calendar is not None:

            # Use only December holidays as anchors
            dec_holidays = [h for h in holiday_dates if h.month == 12]

            t_full_arr     = np.arange(len(df))
            df['expected'] = self._compute_expected(df, t_full_arr)

            biz_day_list = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()
            biz_day_pos  = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}

            for hdate in dec_holidays:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue
                if hdate not in biz_day_pos:
                    continue
                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue
                    tp = h_pos + offset
                    if tp < 0 or tp >= len(biz_day_list):
                        continue
                    target_date = biz_day_list[tp]
                    row         = df.loc[df['Calendar_Date'] == target_date]
                    if row.empty:
                        continue
                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue
                    actual   = row[self.target].values[0]
                    expected = row['expected'].values[0]
                    if expected <= 0 or actual <= 0:
                        continue
                    ratio = actual / expected
                    ratio_accumulator.setdefault(
                        (holiday_dow, offset), []
                    ).append(ratio)

            self.distortion_table_ = {}
            for key, ratios in ratio_accumulator.items():
                if len(ratios) < self.distortion_min_obs:
                    continue
                raw_median = float(np.median(ratios))
                shrunk     = (
                    (1 - self.holiday_shrinkage) * raw_median +
                    self.holiday_shrinkage * 1.0
                )
                capped = float(np.clip(
                    shrunk,
                    1.0 / self.distortion_max_factor,
                    self.distortion_max_factor
                ))
                self.distortion_table_[key] = capped

            print(f"  [Model C] Distortion keys learned: "
                  f"{sorted(self.distortion_table_.keys())}")
        else:
            self.distortion_table_ = {}

        self.fitted = True
        return self

    # ------------------------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit before forecasting.')

        start_date   = pd.to_datetime(start_date)
        end_date     = start_date + pd.DateOffset(months=months_forward)
        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Trend ----
        t_future         = np.arange(self.n_train, self.n_train + len(df))
        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift  = 1 + self.forward_lift_daily * horizon_index

        # ---- Seasonal ----
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, 12), 1.0)
            for d in df['dow']
        ])

        seasonal = dow_e * dom_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)

        forecast = (
            self.base_level * seasonal * trend_multiplier * forward_lift
        )

        # ---- Pre/post cliff scaling ----
        post_cliff_mask = (
            (df['moy'] == 12) & (df['dom'] > self.cliff_date)
        ).values
        forecast[post_cliff_mask] = (
            forecast[post_cliff_mask] *
            (self.cliff_ratio + (1 - self.cliff_ratio) * self.cliff_shrinkage)
        )

        # ---- Weekend hard zero ----
        forecast[df['is_weekend'].values] = 0.0

        # ---- Holiday calendar ----
        if self.master_calendar is not None:
            df = df.merge(self.master_calendar, on='Calendar_Date', how='left')
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

            # Hard zero on observed federal holidays (Dec 25, Jan 1, etc.)
            forecast[df['Is_Holiday'].values == 1] = 0.0

            # ---- Holiday distortion ----
            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )
            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast
            distortion            = np.ones(len(df))

            for hdate in all_relevant_holidays:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue

                    candidate  = hdate
                    steps      = 0
                    direction  = 1 if offset > 0 else -1
                    abs_offset = abs(offset)
                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()
                    if not row_idx:
                        continue
                    idx    = row_idx[0]
                    factor = self.distortion_table_.get(
                        (holiday_dow, offset), 1.0
                    )
                    distortion[idx] = factor

            forecast = forecast * distortion

        df['forecast'] = forecast
        return df

In [11]:
class StructuralLoanForecaster_Switch:
    """
    Switching forecaster that selects Model A, B, or C
    based on the forecast month.

    Rules:
        Month == December              → Model C (December specialist)
        Month != December, >= 22 biz   → Model A
        Month != December, <= 21 biz   → Model B

    Handles partial months correctly: if the forecast starts mid-month,
    it still evaluates the FULL month's business day count to decide
    which model to use, then slices the forecast to only the relevant dates.
    """

    def __init__(self, today, target='total_loan_amount'):
        self.today            = pd.to_datetime(today)
        self.target           = target
        self.model_a          = None
        self.model_b          = None
        self.model_c          = None
        self.holiday_calendar = None
        self.fitted           = False

    @staticmethod
    def _count_biz_days(year, month, holiday_calendar):
        month_start = pd.Timestamp(year=year, month=month, day=1)
        month_end   = month_start + pd.offsets.MonthEnd(0)
        all_days    = pd.date_range(start=month_start, end=month_end, freq='D')
        holidays    = set(pd.to_datetime(
            holiday_calendar.loc[holiday_calendar['Is_Holiday'] == 1, 'Calendar_Date']
        ))
        biz_days = [
            d for d in all_days
            if d.dayofweek not in [5, 6] and d not in holidays
        ]
        return len(biz_days)

    def _select_model(self, year, month):
        if month == 12:
            print(
                f"  [Switch] {year}-{month:02d} → December → Model C selected"
            )
            return 'C', None

        n_biz = self._count_biz_days(year, month, self.holiday_calendar)
        model = 'A' if n_biz >= 22 else 'B'
        print(
            f"  [Switch] {year}-{month:02d} → {n_biz} business days → "
            f"Model {model} selected"
        )
        return model, n_biz

    def fit(self, df, holiday_calendar):
        self.holiday_calendar = holiday_calendar.copy()
        self.holiday_calendar['Calendar_Date'] = pd.to_datetime(
            self.holiday_calendar['Calendar_Date']
        )
        self.model_a = StructuralLoanForecaster_A()
        self.model_b = StructuralLoanForecaster_B()
        self.model_c = StructuralLoanForecaster_C()
        self.model_a.fit(df, holiday_calendar)
        self.model_b.fit(df, holiday_calendar)
        self.model_c.fit(df, holiday_calendar)
        self.fitted = True
        return self

    def forecast_from_date(self, start_date, months_forward=1):
        """
        Generates a forecast from start_date for months_forward months.

        For each calendar month touched by the forecast window:
          1. If December → Model C.
             Otherwise count full month business days → Model A or B.
          2. Run the chosen model for that month.
          3. Slice results to only the dates in [period_start, period_end].
        """
        if not self.fitted:
            raise Exception('Model must be fit before forecasting.')

        start_date = pd.to_datetime(start_date)
        records    = []

        for i in range(months_forward):
            if i == 0:
                period_start = start_date
            else:
                period_start = (start_date + pd.DateOffset(months=i)).replace(day=1)

            month_start = period_start.replace(day=1)
            period_end  = month_start + pd.offsets.MonthEnd(0)

            year  = month_start.year
            month = month_start.month

            model_choice, n_biz = self._select_model(year, month)

            if model_choice == 'A':
                model = self.model_a
            elif model_choice == 'B':
                model = self.model_b
            else:
                model = self.model_c

            pred = model.forecast_from_date(start_date=period_start, months_forward=1)
            pred = pred[
                (pred['Calendar_Date'] >= period_start) &
                (pred['Calendar_Date'] <= period_end)
            ].copy()

            pred['model_used'] = model_choice
            pred['n_biz_days'] = n_biz if n_biz is not None else self._count_biz_days(
                year, month, self.holiday_calendar
            )
            records.append(pred)

        return pd.concat(records, ignore_index=True)

In [12]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   BUSINESS DAY OFFSET HELPER
#////////////////////////////////////////////////////////////////////////////////////////////////////////

def add_business_days(start_date, n_days):
    current    = pd.to_datetime(start_date)
    days_added = 0
    while days_added < n_days:
        current += pd.Timedelta(days=1)
        if current.dayofweek not in [5, 6]:
            days_added += 1
    return current


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   COMPUTE KNOWN WINDOW & FORECAST START
#////////////////////////////////////////////////////////////////////////////////////////////////////////

KNOWN_THROUGH  = add_business_days(TODAY, 3)
FORECAST_START = add_business_days(TODAY, 4)
OUTPUT_THROUGH = add_business_days(TODAY, 29)

print(f"TODAY          : {TODAY.date()}")
print(f"Actuals through: {KNOWN_THROUGH.date()}")
print(f"Forecast start : {FORECAST_START.date()}")
print(f"Output through : {OUTPUT_THROUGH.date()}")


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   BUILD HOLIDAY CALENDAR
#////////////////////////////////////////////////////////////////////////////////////////////////////////

holiday_calendar = build_us_holiday_calendar(
    start_date='2021-01-01',
    end_date='2030-12-31'
)


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   TRAIN DATA (through KNOWN_THROUGH)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

train_wholesale = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= '2021-09-01') &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
].drop(columns=['Is_Holiday'], errors='ignore').copy()

train_wholesale['Calendar_Date'] = pd.to_datetime(train_wholesale['Calendar_Date'])


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   FIT SWITCH MODEL & FORECAST
#                                   months_forward=2 ensures we always cover a full
#                                   30-business-day window even when starting mid-month
#////////////////////////////////////////////////////////////////////////////////////////////////////////

model_switch = StructuralLoanForecaster_Switch(today=FORECAST_START)
model_switch.fit(train_wholesale, holiday_calendar)

pred_switch = model_switch.forecast_from_date(
    start_date=FORECAST_START,
    months_forward=2          # ← covers partial current month + next full month
)
pred_switch = pred_switch[pred_switch['Calendar_Date'] <= OUTPUT_THROUGH].copy()

#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PULL ACTUALS FOR KNOWN WINDOW
#////////////////////////////////////////////////////////////////////////////////////////////////////////

actuals_window = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
][['Calendar_Date', 'total_loan_amount']].copy()

actuals_window['Calendar_Date'] = pd.to_datetime(actuals_window['Calendar_Date'])
actuals_window = actuals_window.rename(columns={'total_loan_amount': 'amount'})
actuals_window['source']     = 'actual'
actuals_window['model_used'] = '—'
actuals_window['forecast']   = None


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   BUILD COMBINED OUTPUT
#////////////////////////////////////////////////////////////////////////////////////////////////////////

forecast_out = pred_switch[['Calendar_Date', 'forecast', 'model_used']].copy()
forecast_out['amount'] = None
forecast_out['source'] = 'forecast'

wholesale_combined = pd.concat(
    [actuals_window, forecast_out], ignore_index=True
).sort_values('Calendar_Date').reset_index(drop=True)

wholesale_combined['channel'] = 'Wholesale'

# Business days only
wholesale_combined = wholesale_combined[
    ~wholesale_combined['Calendar_Date'].dt.dayofweek.isin([5, 6])
].reset_index(drop=True)


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PRINT OUTPUT
#////////////////////////////////////////////////////////////////////////////////////////////////////////

dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '         —'
    return f'{float(val):>12,.0f}'

header = (
    f"{'Date':<15} {'DOW':<6} {'Source':<10} {'Model':<8} "
    f"{'Channel':<14} {'Amount':>12}"
)
print(f"\n{header}")
print("-" * 70)

for _, row in wholesale_combined.iterrows():
    dow_label = dow_names.get(row['Calendar_Date'].dayofweek, '—')
    amount    = row['amount'] if row['source'] == 'actual' else row['forecast']
    print(
        f"{str(row['Calendar_Date'].date()):<15} "
        f"{dow_label:<6} "
        f"{row['source']:<10} "
        f"{str(row['model_used']):<8} "
        f"{row['channel']:<14} "
        f"{fmt(amount)}"
    )

print("-" * 70)

actual_total   = actuals_window['amount'].dropna().astype(float).sum()
forecast_total = pred_switch['forecast'].dropna().astype(float).sum()

print(
    f"{'TOTAL (actuals + forecast)':<46} "
    f"{fmt(actual_total + forecast_total)}"
)

TODAY          : 2026-03-10
Actuals through: 2026-03-13
Forecast start : 2026-03-16
Output through : 2026-04-20
  [Model C] cliff_ratio (post/pre Dec 18): 0.917
  [Model C] Distortion keys learned: [(0, -2), (0, -1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 6), (4, -2), (4, -1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 6)]
  [Switch] 2026-03 → 22 business days → Model A selected
  [Switch] 2026-04 → 22 business days → Model A selected

Date            DOW    Source     Model    Channel              Amount
----------------------------------------------------------------------
2026-03-10      Tue    actual     —        Wholesale         8,398,873
2026-03-11      Wed    actual     —        Wholesale        11,053,901
2026-03-12      Thu    actual     —        Wholesale         2,016,338
2026-03-13      Fri    actual     —        Wholesale         8,873,504
2026-03-16      Mon    forecast   A        Wholesale        14,764,677
2026-03-17      Tue    forecast   A        Wholesale         9,003,459
20

In [13]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   REBUILD EVAL_DF TO COVER FULL REQUESTED DATE RANGE
#////////////////////////////////////////////////////////////////////////////////////////////////////////

VIEW_FROM = pd.to_datetime(TODAY)
VIEW_TO   = pd.to_datetime('2026-04-10')

print(f"Requested range: {VIEW_FROM.date()} → {VIEW_TO.date()}")

# ── Pull actuals for the full requested window ────────────────────────────────
full_actuals = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= VIEW_TO)
][['Calendar_Date', 'total_loan_amount']].copy()

full_actuals['Calendar_Date'] = pd.to_datetime(full_actuals['Calendar_Date'])
full_actuals = full_actuals.rename(columns={'total_loan_amount': 'actual'})

# ── Re-fit switch model ─────────���─────────────────────────────────────────────
model_switch_eval = StructuralLoanForecaster_Switch(today=FORECAST_START)
model_switch_eval.fit(
    wholesale_historical[
        (wholesale_historical['Calendar_Date'] >= '2021-09-01') &
        (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
    ].drop(columns=['Is_Holiday'], errors='ignore').copy(),
    holiday_calendar
)

# months_forward=2 ensures we always cover VIEW_TO even for mid-month starts
months_needed = (
    (VIEW_TO.year - FORECAST_START.year) * 12 +
    (VIEW_TO.month - FORECAST_START.month) + 2
)

pred_switch_eval = model_switch_eval.forecast_from_date(
    start_date=FORECAST_START,
    months_forward=months_needed
)

# Cap to VIEW_TO
pred_switch_eval = pred_switch_eval[pred_switch_eval['Calendar_Date'] <= VIEW_TO].copy()

# ── Build eval_df ─────────────────────────────────────────────────────────────
eval_df = full_actuals.merge(
    pred_switch_eval[['Calendar_Date', 'forecast', 'model_used']],
    on='Calendar_Date', how='inner'
)

eval_df['channel'] = 'Wholesale'
eval_df['dow']     = eval_df['Calendar_Date'].dt.dayofweek

# Business days with actual > 0 only
eval_df = eval_df[
    (~eval_df['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (eval_df['actual'] > 0)
].reset_index(drop=True)

# ── Known window: replace forecast with actuals ───────────────────────────────
known_mask = eval_df['Calendar_Date'] <= KNOWN_THROUGH
eval_df.loc[known_mask, 'forecast']   = eval_df.loc[known_mask, 'actual']
eval_df.loc[known_mask, 'model_used'] = 'actual'

eval_df['diff']    = eval_df['forecast'] - eval_df['actual']
eval_df['pct_err'] = (eval_df['diff'] / eval_df['actual']) * 100

eval_df['source'] = 'forecast'
eval_df.loc[known_mask, 'source'] = 'actual'

print(f"eval_df covers: {eval_df['Calendar_Date'].min().date()} → {eval_df['Calendar_Date'].max().date()}")
print(f"eval_df rows  : {len(eval_df)}")


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PRINT OUTPUT TABLE
#////////////////////////////////////////////////////////////////////////////////////////////////////////

dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):,.0f}'

def fmt_pct(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):+.1f}%'

# ── Known actuals window ──────────────────────────────────────────────────────
known_actuals_print = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH) &
    (~wholesale_historical['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (wholesale_historical['total_loan_amount'] > 0)
][['Calendar_Date', 'total_loan_amount']].copy()

known_actuals_print['Calendar_Date'] = pd.to_datetime(known_actuals_print['Calendar_Date'])
known_actuals_print = known_actuals_print.rename(columns={'total_loan_amount': 'actual'})
known_actuals_print['source']    = 'actual'
known_actuals_print['channel']   = 'Wholesale'
known_actuals_print['forecast']  = known_actuals_print['actual']
known_actuals_print['model_used']= 'actual'
known_actuals_print['diff']      = 0.0
known_actuals_print['pct_err']   = 0.0

# ── Forecast rows ─────────────────────────────────────────────────────────────
forecast_rows = eval_df[eval_df['source'] == 'forecast'].copy()

# ── Combine ───────────────────────────────────────────────────────────────────
print_df = pd.concat(
    [known_actuals_print, forecast_rows], ignore_index=True
).sort_values(
    ['Calendar_Date', 'source'], ascending=[True, True]
).drop_duplicates(
    subset='Calendar_Date', keep='first'
).reset_index(drop=True)

# ── Apply date band filter ────────────────────────────────────────────────────
print_df_view = print_df[
    (print_df['Calendar_Date'] >= VIEW_FROM) &
    (print_df['Calendar_Date'] <= VIEW_TO)
].copy().reset_index(drop=True)

# ── Totals & MAE/MAPE ─────────────────────────────────────────────────────────
forecast_mask_view = print_df_view['source'] == 'forecast'

total_actual   = print_df_view.loc[forecast_mask_view, 'actual'].sum()
total_forecast = print_df_view.loc[forecast_mask_view, 'forecast'].sum()
total_diff     = total_forecast - total_actual
total_pct      = (total_diff / total_actual) * 100 if total_actual > 0 else float('nan')

mae  = print_df_view.loc[forecast_mask_view, 'diff'].abs().mean()    if forecast_mask_view.any() else float('nan')
mape = print_df_view.loc[forecast_mask_view, 'pct_err'].abs().mean() if forecast_mask_view.any() else float('nan')

# ── Build table columns ───────────────────────────────────────────────────────
col_date      = []
col_dow       = []
col_source    = []
col_model     = []
col_channel   = []
col_actual    = []
col_forecast  = []
col_diff      = []
col_pct       = []

for _, row in print_df_view.iterrows():
    col_date.append(str(row['Calendar_Date'].date()))
    col_dow.append(dow_names.get(row['Calendar_Date'].dayofweek, '—'))
    col_source.append(row['source'])
    col_model.append(str(row['model_used']))
    col_channel.append(row['channel'])
    col_actual.append(fmt(row['actual']))
    col_forecast.append(fmt(row['forecast']))
    col_diff.append(fmt(row['diff']))
    col_pct.append(fmt_pct(row['pct_err']))

# ── TOTAL row ─────────────────────────────────────────────────────────────────
col_date.append(f'TOTAL  ({VIEW_FROM.strftime("%b %d")} → {VIEW_TO.strftime("%b %d")})')
col_dow.append('')
col_source.append('forecast only')
col_model.append('')
col_channel.append('Wholesale')
col_actual.append(fmt(total_actual))
col_forecast.append(fmt(total_forecast))
col_diff.append(fmt(total_diff))
col_pct.append(fmt_pct(total_pct))

# ── MAE row ───────────────────────────────────────────────────────────────────
col_date.append('MAE')
for lst in [col_dow, col_source, col_model, col_channel, col_actual]:
    lst.append('')
col_forecast.append(fmt(mae))
col_diff.append('')
col_pct.append('')

# ── MAPE row ──────────────────────────────────────────────────────────────────
col_date.append('MAPE')
for lst in [col_dow, col_source, col_model, col_channel, col_actual]:
    lst.append('')
col_forecast.append(fmt_pct(mape))
col_diff.append('')
col_pct.append('')

# ── Row fill colours ──────────────────────────────────────────────────────────
n_known_view    = print_df_view[print_df_view['source'] == 'actual'].shape[0]
n_forecast_view = print_df_view[print_df_view['source'] == 'forecast'].shape[0]
n_summary       = 3

fill_colors = (
    ['#f0f0f0'] * n_known_view    +
    ['white']   * n_forecast_view +
    ['#fffbea'] * n_summary
)

# ── Plotly table ──────────────────────────────────────────────────────────────
table_fig = go.Figure(data=[go.Table(
    columnwidth = [110, 50, 80, 60, 80, 100, 100, 100, 80],
    header = dict(
        values = [
            '<b>Date</b>', '<b>DOW</b>', '<b>Source</b>', '<b>Model</b>',
            '<b>Channel</b>', '<b>Actual</b>',
            '<b>Forecast</b>', '<b>Diff</b>', '<b>Pct</b>'
        ],
        fill_color = '#2a3f5f',
        font       = dict(color='white', size=11),
        align      = ['left', 'center', 'center', 'center', 'center',
                      'right', 'right', 'right', 'right'],
        height     = 30
    ),
    cells = dict(
        values = [
            col_date, col_dow, col_source, col_model, col_channel,
            col_actual, col_forecast, col_diff, col_pct
        ],
        fill_color = [fill_colors] * 9,
        font       = dict(color='black', size=11),
        align      = ['left', 'center', 'center', 'center', 'center',
                      'right', 'right', 'right', 'right'],
        height     = 26
    )
)])

table_fig.update_layout(
    title = dict(
        text = (
            f'Wholesale — Actuals vs Forecast | '
            f'{VIEW_FROM.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}'
        ),
        font = dict(size=13)
    ),
    width  = 1000,
    height = 80 + (n_known_view + n_forecast_view + n_summary) * 28,
    margin = dict(l=10, r=10, t=50, b=10)
)

table_fig.show()


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PLOT — ACTUALS vs FORECAST (SWITCH MODEL)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

fig = go.Figure()

# ── Actuals line ──────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x      = print_df['Calendar_Date'],
    y      = print_df['actual'],
    mode   = 'lines+markers',
    name   = 'Actual',
    line   = dict(color='black', width=2),
    marker = dict(size=6)
))

# ── Forecast line (colour-coded by model) ─────────────────────────────────────
forecast_only = print_df[print_df['source'] == 'forecast'].copy()

for model_label, color in [('A', 'steelblue'), ('B', 'darkorange')]:
    mask = forecast_only['model_used'] == model_label
    if mask.any():
        fig.add_trace(go.Scatter(
            x      = forecast_only.loc[mask, 'Calendar_Date'],
            y      = forecast_only.loc[mask, 'forecast'],
            mode   = 'lines+markers',
            name   = f'Forecast (Model {model_label})',
            line   = dict(color=color, width=2.5, dash='dash'),
            marker = dict(size=6)
        ))

fig.add_vline(
    x          = str(FORECAST_START.date()),
    line_width = 1.5,
    line_dash  = 'dash',
    line_color = 'grey'
)
fig.add_annotation(
    x=str(FORECAST_START.date()), y=1, yref='paper',
    text='Forecast Start', showarrow=False,
    xanchor='left', font=dict(color='grey', size=11)
)
fig.add_vrect(
    x0=str(TODAY.date()), x1=str(KNOWN_THROUGH.date()),
    fillcolor='lightgrey', opacity=0.2, layer='below', line_width=0
)
fig.add_annotation(
    x=str(TODAY.date()), y=1, yref='paper',
    text='Known Actuals', showarrow=False,
    xanchor='left', font=dict(color='grey', size=11)
)

fig.update_layout(
    title = dict(
        text=(
            f'Wholesale — Actuals vs Forecast (Switch Model)<br>'
            f'<sup>{TODAY.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}</sup>'
        ),
        font=dict(size=18)
    ),
    xaxis=dict(title='Date', tickformat='%b %d', tickangle=-45,
               showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(title='Loan Volume ($)', tickformat='$,.0f',
               showgrid=True, gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    hovermode='x unified', plot_bgcolor='white', width=1200, height=550
)

fig.show()

Requested range: 2026-03-10 → 2026-04-10
  [Model C] cliff_ratio (post/pre Dec 18): 0.917
  [Model C] Distortion keys learned: [(0, -2), (0, -1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 6), (4, -2), (4, -1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 6)]
  [Switch] 2026-03 → 22 business days → Model A selected
  [Switch] 2026-04 → 22 business days → Model A selected
  [Switch] 2026-05 → 20 business days → Model B selected
eval_df covers: 2026-03-16 → 2026-04-10
eval_df rows  : 20


Model A Back Testing

In [47]:
import pandas as pd
import numpy as np
from itertools import product

def tune_model_a_by_month(
    historical_df,
    holiday_calendar,
    backtest_start='2024-03-01',
    backtest_end='2026-03-31',
    train_min_start='2021-09-01',
    target='total_loan_amount',
):
    """
    For EVERY month in [backtest_start, backtest_end]:
      - Try every combination in the param grid with Model A
      - Pick the config that minimizes abs(monthly_pct_err)
      - Report results
    """

    param_grid = {
        'recency_strength':      [0.35, 1.0],
        'dow_shrinkage':         [0.05, 0.25],
        'interaction_shrinkage': [0.20, 0.35],
        'seasonal_exponent':     [1.0, 1.02],
        'trend_dampening':       [0.7, 0.83],
        'forward_lift_daily':    [0.0, 0.0025],
        'level_lookback_days':   [60],
        'trend_seasonal_tilt':   [0.0, 0.08],
    }

    param_keys = list(param_grid.keys())
    param_combos = list(product(*param_grid.values()))
    print(f"Total param combos to test per month: {len(param_combos)}")

    periods = pd.date_range(
        start=pd.to_datetime(backtest_start).replace(day=1),
        end=pd.to_datetime(backtest_end).replace(day=1),
        freq='MS'
    )

    all_results = []

    for period_start in periods:
        year = period_start.year
        month = period_start.month

        period_end = period_start + pd.offsets.MonthEnd(0)
        train_cutoff = period_start - pd.Timedelta(days=1)

        train_df = historical_df[
            (historical_df['Calendar_Date'] >= train_min_start) &
            (historical_df['Calendar_Date'] <= train_cutoff)
        ].drop(columns=['Is_Holiday'], errors='ignore').copy()

        if len(train_df) < 100:
            print(f"  Skipping {year}-{month:02d}: insufficient training data")
            continue

        actuals = historical_df[
            (historical_df['Calendar_Date'] >= period_start) &
            (historical_df['Calendar_Date'] <= period_end) &
            (~historical_df['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
            (historical_df[target] > 0)
        ][['Calendar_Date', target]].copy()

        if actuals.empty:
            continue

        actual_total = actuals[target].sum()

        print(f"\n{'='*70}")
        print(f"  Tuning {year}-{month:02d}  |  Actual total: ${actual_total:,.0f}")
        print(f"{'='*70}")

        best_err = float('inf')
        best_params = None
        best_forecast_total = None

        for combo in param_combos:
            params = dict(zip(param_keys, combo))
            try:
                m = StructuralLoanForecaster_A(target=target, **params)
                m.fit(train_df, holiday_calendar)
                pred = m.forecast_from_date(
                    start_date=period_start, months_forward=1
                )
                merged = actuals.merge(
                    pred[['Calendar_Date', 'forecast']],
                    on='Calendar_Date', how='inner'
                )
                if merged.empty:
                    continue

                forecast_total = merged['forecast'].sum()
                pct_err = abs(forecast_total - actual_total) / actual_total * 100

                if pct_err < best_err:
                    best_err = pct_err
                    best_params = params.copy()
                    best_forecast_total = forecast_total

            except Exception:
                continue

        if best_params is not None:
            signed_err = (best_forecast_total - actual_total) / actual_total * 100
            print(f"  BEST  |  Monthly err: {signed_err:+.2f}%  |  Abs err: {best_err:.2f}%")
            for k, v in best_params.items():
                print(f"    {k:<28s} = {v}")

            all_results.append({
                'year': year,
                'month': month,
                'period': f"{year}-{month:02d}",
                'actual_total': actual_total,
                'forecast_total': best_forecast_total,
                'monthly_pct_err': signed_err,
                'abs_pct_err': best_err,
                **best_params,
            })
        else:
            print(f"  No valid config found for {year}-{month:02d}")

    # ── Summary ──────────────────────────────────────────────────────────
    results_df = pd.DataFrame(all_results)

    print(f"\n{'='*70}")
    print("  OVERALL SUMMARY — Best Model A Config Per Month")
    print(f"{'='*70}")
    print(results_df[['period', 'monthly_pct_err', 'abs_pct_err']].to_string(index=False))
    print(f"\nMean abs monthly error : {results_df['abs_pct_err'].mean():.2f}%")
    print(f"Median abs monthly error: {results_df['abs_pct_err'].median():.2f}%")
    print(f"Within ±5%             : {(results_df['abs_pct_err'] <= 5).sum()}/{len(results_df)}")
    print(f"Within ±10%            : {(results_df['abs_pct_err'] <= 10).sum()}/{len(results_df)}")

    # ── Which params appear most often in winning configs? ───────────────
    print(f"\n{'='*70}")
    print("  MOST COMMON WINNING PARAMETER VALUES")
    print(f"{'='*70}")
    for key in param_keys:
        counts = results_df[key].value_counts()
        print(f"\n  {key}:")
        for val, cnt in counts.items():
            print(f"    {val:<10}  appeared in {cnt}/{len(results_df)} months")

    return results_df

In [49]:
# Build holiday calendar
holiday_calendar = build_us_holiday_calendar('2021-01-01', '2030-12-31')

# Run the tuning
tuning_results = tune_model_a_by_month(
    historical_df    = wholesale_historical,
    holiday_calendar = holiday_calendar,
    backtest_start   = '2026-02-01',
    backtest_end     = '2026-03-31',
    train_min_start  = '2021-09-01',
    target           = 'total_loan_amount',
)

# Save to CSV if you want
# tuning_results.to_csv('model_a_tuning_results.csv', index=False)

Total param combos to test per month: 128

  Tuning 2026-02  |  Actual total: $205,220,387
  BEST  |  Monthly err: -0.09%  |  Abs err: 0.09%
    recency_strength             = 1.0
    dow_shrinkage                = 0.25
    interaction_shrinkage        = 0.35
    seasonal_exponent            = 1.02
    trend_dampening              = 0.83
    forward_lift_daily           = 0.0
    level_lookback_days          = 60
    trend_seasonal_tilt          = 0.0

  Tuning 2026-03  |  Actual total: $246,341,013
  BEST  |  Monthly err: -0.47%  |  Abs err: 0.47%
    recency_strength             = 1.0
    dow_shrinkage                = 0.05
    interaction_shrinkage        = 0.2
    seasonal_exponent            = 1.02
    trend_dampening              = 0.83
    forward_lift_daily           = 0.0
    level_lookback_days          = 60
    trend_seasonal_tilt          = 0.08

  OVERALL SUMMARY — Best Model A Config Per Month
 period  monthly_pct_err  abs_pct_err
2026-02        -0.086464     0.086464
202

model A backtesting intra month

In [23]:
import pandas as pd
import numpy as np
from itertools import product

def tune_model_a_15th_to_15th(
    historical_df,
    holiday_calendar,
    period_start_date='2025-11-15',
    train_min_start='2021-09-01',
    target='total_loan_amount',
):
    """
    Test all param combos for a single 15th-to-15th period.
    period_start_date should be a 15th (e.g. '2026-03-15').
    Forecast window: 15th → 15th of next month.
    Training cutoff: period_start - 1 day (no lookahead).
    """

    param_grid = {
        'recency_strength':      [0.2, 0.35, 1.0],
        'dow_shrinkage':         [0.05, 0.25, 0.1],
        'interaction_shrinkage': [0.20, 0.35, 0.5],
        'seasonal_exponent':     [1.0, 1.02, 1.05],
        'trend_dampening':       [0.7, 0.83, 0.9],
        'forward_lift_daily':    [0.0, 0.0025, 0.005],
        'level_lookback_days':   [30, 60, 90],
        'trend_seasonal_tilt':   [0.0, 0.08, 0.15],
    }

    param_keys = list(param_grid.keys())
    param_combos = list(product(*param_grid.values()))
    print(f"Total param combos to test: {len(param_combos)}")

    period_start = pd.to_datetime(period_start_date)
    period_end   = period_start + pd.DateOffset(months=1)  # 15th of next month

    train_cutoff = period_start - pd.Timedelta(days=1)

    print(f"Period        : {period_start.date()} → {period_end.date()}")
    print(f"Train cutoff  : {train_cutoff.date()}")

    train_df = historical_df[
        (historical_df['Calendar_Date'] >= train_min_start) &
        (historical_df['Calendar_Date'] <= train_cutoff)
    ].drop(columns=['Is_Holiday'], errors='ignore').copy()

    if len(train_df) < 100:
        print(f"  ERROR: insufficient training data ({len(train_df)} rows)")
        return None

    actuals = historical_df[
        (historical_df['Calendar_Date'] >= period_start) &
        (historical_df['Calendar_Date'] <= period_end) &
        (~historical_df['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
        (historical_df[target] > 0)
    ][['Calendar_Date', target]].copy()

    if actuals.empty:
        print("  ERROR: no actuals found in this window")
        return None

    actual_total = actuals[target].sum()

    print(f"Actual biz days: {len(actuals)}")
    print(f"Actual total   : ${actual_total:,.0f}")

    # ── Need 2 months_forward since period spans 2 calendar months ────────
    best_err = float('inf')
    best_params = None
    best_forecast_total = None

    for combo in param_combos:
        params = dict(zip(param_keys, combo))
        try:
            m = StructuralLoanForecaster_A(target=target, **params)
            m.fit(train_df, holiday_calendar)

            # Forecast 2 months from period_start to cover the full 15th→15th
            pred = m.forecast_from_date(
                start_date=period_start, months_forward=2
            )
            # Trim to our exact window
            pred = pred[
                (pred['Calendar_Date'] >= period_start) &
                (pred['Calendar_Date'] <= period_end)
            ]

            merged = actuals.merge(
                pred[['Calendar_Date', 'forecast']],
                on='Calendar_Date', how='inner'
            )
            if merged.empty:
                continue

            forecast_total = merged['forecast'].sum()
            pct_err = abs(forecast_total - actual_total) / actual_total * 100

            if pct_err < best_err:
                best_err = pct_err
                best_params = params.copy()
                best_forecast_total = forecast_total

        except Exception:
            continue

    if best_params is not None:
        signed_err = (best_forecast_total - actual_total) / actual_total * 100

        print(f"\n{'='*70}")
        print(f"  BEST PARAMS for {period_start.date()} → {period_end.date()}")
        print(f"{'='*70}")
        print(f"  Signed err : {signed_err:+.2f}%")
        print(f"  Abs err    : {best_err:.2f}%")
        print(f"  Actual     : ${actual_total:,.0f}")
        print(f"  Forecast   : ${best_forecast_total:,.0f}")
        print()
        for k, v in best_params.items():
            print(f"    {k:<28s} = {v}")

        return {
            'period': f"{period_start.date()} → {period_end.date()}",
            'actual_total': actual_total,
            'forecast_total': best_forecast_total,
            'pct_err': signed_err,
            'abs_pct_err': best_err,
            **best_params,
        }
    else:
        print(f"  No valid config found")
        return None

In [24]:
result = tune_model_a_15th_to_15th(
    wholesale_historical,
    holiday_calendar,
    period_start_date='2025-11-15',
)

Total param combos to test: 6561
Period        : 2025-11-15 → 2025-12-15
Train cutoff  : 2025-11-14
Actual biz days: 20
Actual total   : $220,485,251

  BEST PARAMS for 2025-11-15 → 2025-12-15
  Signed err : +0.01%
  Abs err    : 0.01%
  Actual     : $220,485,251
  Forecast   : $220,497,065

    recency_strength             = 0.35
    dow_shrinkage                = 0.1
    interaction_shrinkage        = 0.5
    seasonal_exponent            = 1.0
    trend_dampening              = 0.83
    forward_lift_daily           = 0.0025
    level_lookback_days          = 90
    trend_seasonal_tilt          = 0.0
